# Setup Conda

In [0]:
import subprocess
import os
import json
import pathlib
from pathlib import Path

In [0]:
# import subprocess
# import json
# from pathlib import Path

envs_to_export = [
    "vdeidtorch", #torch, transformer, PEFT
    #"vdeidspacy", #spacy
    "vdeidpresidio", # presidio
]

def conda_json(cmd):
    result = subprocess.run(
        ["conda", *cmd, "--json"],
        capture_output=True,
        text=True,
        check=True
    )
    return json.loads(result.stdout)

def purge_selected_conda_envs():
    try:
        # Get all env paths
        env_data = conda_json(["info", "--envs"])
        env_paths = env_data.get("envs", [])

        # Get base/root prefix
        info_data = conda_json(["info"])
        base_path = Path(info_data.get("root_prefix")).resolve()

        print(f"Base path identified as: {base_path}")

        for path in env_paths:
            env_path = Path(path).resolve()
            env_name = env_path.name

            # 🔒 Never touch base
            if env_path == base_path:
                print("Skipping base environment")
                continue

            # Only remove envs explicitly listed
            if env_name not in envs_to_export:
                print(f"Skipping environment: {env_name}")
                continue

            print(f"Removing environment '{env_name}' at: {env_path}")
            subprocess.run(
                ["conda", "remove", "-p", str(env_path), "--all", "-y"],
                check=True
            )

        print("\nSelected environment cleanup complete.")

    except subprocess.CalledProcessError as e:
        print(f"Error: Conda command failed: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    purge_selected_conda_envs()

In [0]:
# setup venvs
# List of commands to execute

commands = [
    # ===== conda env setup =====
    "conda create -n vdeidtorch python=3.11 -y",
    # "conda create -n vdeidspacy python=3.11 -y",
    "conda create -n vdeidpresidio python=3.11 -y",
    
    # ===== vdeidspacy =====
    # The ner model needs spacy >=3.5.x < 3.6
    # Core: keep spaCy + compiled deps consistent (conda-forge)
    # 'conda run -n vdeidspacy conda install -y -c conda-forge "spacy==3.5.4" "numpy<2.0" "thinc>=8.1,<8.2" "cython<3" "cymem" "preshed" "murmurhash" "blis" --update-deps --force-reinstall',
    # Transformers integration (prefer conda-forge to avoid pip/ABI mismatch) 
    # 'conda run -n vdeidspacy conda install -y -c conda-forge spacy-transformers --update-deps --force-reinstall',
    # 'conda run -n vdeidspacy conda install -y -c pytorch -c nvidia pytorch pytorch-cuda=11.8',
    #  DO NOT install thinc[cuda11x] (brings CuPy/ABI headaches on Windows) 
    # 'conda run -n vdeidspacy python -m pip install -U "thinc[cuda11x]"',
    # Install the Danish spaCy model wheel (model-only; OK via pip) 
    # "mv da_dacy_large_ner_fine_grained-any-py3-none-any.whl da_dacy_large_ner_fine_grained-0.0.1-py3-none-any.whl", # for linux/mac
    # 'rename da_dacy_large_ner_fine_grained-any-py3-none-any.whl da_dacy_large_ner_fine_grained-0.0.1-py3-none-any.whl',
    # 'conda run -n vdeidspacy python -m pip install --no-cache-dir da_dacy_large_ner_fine_grained-0.0.1-py3-none-any.whl',

    #  Remove the wheel file after install (Windows PowerShell / CMD) 
    # If you run from PowerShell:
    # 'powershell -NoProfile -Command "Remove-Item -Force da_dacy_large_ner_fine_grained-0.0.1-py3-none-any.whl"',
    # If you run from CMD instead, use this (comment out the PowerShell line above):
    #'del /f /q da_dacy_large_ner_fine_grained-0.0.1-py3-none-any.whl',

    # the srsls package version should be specified
    'conda run -n vdeidspacy python -m pip install --no-deps --force-reinstall srsly==2.4.8',
    
    # ===== vdeidtorch =====
    #"conda run -n vdenotransdeidai pip3 install -U torch torchaudio", # for mac
    "conda run -n vdeidtorch pip3 install -U torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118",
    "conda run -n vdeidtorch pip3 install transformers nervaluate datasets",
    "conda run -n vdeidtorch pip3 install pyctcdecode",
    "conda run -n vdeidtorch conda install -c conda-forge peft",

    # ===== vdeidpresidio =====
    "conda run -n vdeidpresidio pip3 install presidio-analyzer presidio-anonymizer",
]

def run_commands(command_list):
    for cmd in command_list:
        print(f"Executing: {cmd}")
        try:
            # shell=True is required for conda commands as they are often shell functions/aliases
            subprocess.run(cmd, shell=True, check=True)
            print(f"Successfully executed: {cmd}\n")
        except subprocess.CalledProcessError as e:
            print(f"Error occurred while executing {cmd}: {e}\n")

if __name__ == "__main__":
    run_commands(commands)


In [0]:
# print the package info for replication

# import subprocess
# import json
# import pathlib
# import os


# List of environments to export
envs_to_export = [
    "vdeidtorch",
    "vdeidpresidio", 
    "vdeidspacy",
]

# Output directory
output_dir = pathlib.Path("condaenv_packages_versions")
output_dir.mkdir(exist_ok=True)

# Capture the current system environment dictionary
# This ensures subprocess can find 'conda' and other system tools
current_env = os.environ.copy()

for env_name in envs_to_export:
    print(f"Exporting packages for env: {env_name}")

    try:
        # We use env_name (string) in the command list
        # We use current_env (dictionary) for the env parameter
        result = subprocess.run(
            ["conda", "list", "-n", env_name, "--json"],
            capture_output=True,
            text=True,
            check=True,
            env=current_env  # Pass the dictionary here
        )

        packages = json.loads(result.stdout)

        # Write to <env_name>.json
        output_file = output_dir / f"{env_name}.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(packages, f, indent=2)
            
    except subprocess.CalledProcessError as e:
        print(f"❌ Error exporting {env_name}: {e}")
    except Exception as e:
        print(f"⚠️ Unexpected error for {env_name}: {e}")

print("✅ Done. JSON files saved.")

# Preproc-Read the alrttm files

In [0]:
import glob
import os

folder = "pseudonymization_schema_aligned/parent_labels/"

keep_numbers = ("94", "95", "105", "169", "92", "17")

files = [
    f for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
    if (
        os.path.isfile(f)
        and any(x in os.path.basename(f).lower() for x in ("recheck", "consensus"))
        and any(x in os.path.basename(f) for x in keep_numbers)
    )
]

print(files)

## remove the PII human annos
This is for non-training PII pipe.

In [0]:
import glob
import os
import re

folder = "pseudonymization_schema_aligned/parent_labels/"

keep_numbers = ("94", "95", "105", "169", "92", "17")

files = [
    f for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
    if (
        os.path.isfile(f)
        and any(x in os.path.basename(f).lower() for x in ("recheck", "consensus"))
        and any(x in os.path.basename(f) for x in keep_numbers)
    )
]

# Remove anything enclosed in square brackets, including the brackets.
# Examples:
# [PersonData] -> ""
# [MISC]       -> ""
# [PER]        -> ""
# [NEW_LABEL]  -> ""
bracket_pattern = re.compile(r"\[[^\[\]]*\]")

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    cleaned_text = bracket_pattern.sub("", text)

    # Overwrite original file
    with open(file, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned: {file}")

# pseudopipe

## 1st presidio

In [0]:
import os
import subprocess

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SCRIPT = "pseudopipe/pseudopipe_deidentify_danish_presidio.py"

# Folder containing the original transcripts.
# The files DO NOT need pre-existing PII labels.
INPUT_DIR = "pseudonymization_schema_aligned/child_labels"

# Where de-identified transcripts and audit files will be written.
OUTPUT_DIR =  "annonydata/presidio_deidentified"


# ------------------------------------------------------------
# Call the script inside vdeidpresidio
# ------------------------------------------------------------

cmd = [
    "conda",
    "run",
    "--no-capture-output",
    "-n",
    "vdeidpresidio",
    "python",
    SCRIPT,

    "--input-dir",
    INPUT_DIR,

    "--output-dir",
    OUTPUT_DIR,

    # Recursively process both transcript formats
    "--extensions",
    ".txt",
    ".alfrttm",

    # Keep fairly high recall for PII detection
    "--min-score",
    "0.05",

    # Makes the random replacements reproducible.
    # Change/remove this if you want a different randomization.
    "--seed",
    "42",
]

print("Running:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    cwd=os.getcwd(),
    check=True,
)

print("\nPresidio de-identification finished.")

## 2nd PerLocOrg

In [0]:
import os
import subprocess
from pathlib import Path


# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

SCRIPT = (
    PROJECT_ROOT
    / "pseudopipe"
    / "pseudopipe_identify_danish_pii_ner.py"
)

# Stage 1: Presidio output
INPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "presidio_deidentified"
    / "deidentified"
)

# Stage 2 output
OUTPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "danish_per_loc_org_pii"
)


# ============================================================
# Sanity checks
# ============================================================

if not SCRIPT.is_file():
    raise FileNotFoundError(f"Script not found:\n{SCRIPT}")

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input directory not found:\n{INPUT_DIR}")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Run Stage 2 in vdeidtorch
# ============================================================

cmd = [
    "conda",
    "run",
    "--no-capture-output",
    "-n",
    "vdeidtorch",
    "python",
    str(SCRIPT),

    "--input-dir",
    str(INPUT_DIR),

    "--output-dir",
    str(OUTPUT_DIR),

    "--extensions",
    ".txt",
    ".alfrttm",

    # High-recall PII screening
    "--min-score",
    "0.0",

    # Long-input overlap
    "--stride",
    "128",

    # --------------------------------------------------------
    # Output format:
    #
    # [PersonData]Peter Jensen[PER]
    # [PersonData]Novo Nordisk[ORG]
    # [PersonData]København[LOC]
    # --------------------------------------------------------
    "--annotation-marker",
    "PersonData",

    # ALFRRTM lines must contain a quoted utterance.
    # start/stop/speaker metadata is NEVER passed to XLM-R.
    "--strict-quoted-utterance",
]


print("Running:")
print(" ".join(cmd))
print()


subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    check=True,
)


print("\nStage-2 PII identification finished.")

## 3rd More NER and MISC
keep the NER except MISC from previous, also deal with MISC

In [0]:
import os
import subprocess
from pathlib import Path


# ============================================================
# Project root
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()


# ============================================================
# Configuration
# ============================================================

SCRIPT = (
    PROJECT_ROOT
    / "pseudopipe"
    / "pseudopipe_identify_danish_pii_spacy.py"
)

# Stage-2 tagged transcripts
INPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "danish_per_loc_org_pii"
    / "tagged"
)

# Stage-3 output
OUTPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "dacy_finegrained_pii"
)


# ============================================================
# Sanity checks
# ============================================================

print("Current working directory:")
print(PROJECT_ROOT)

print("\nScript:")
print(SCRIPT)

print("\nStage-3 input:")
print(INPUT_DIR)

print("\nStage-3 output:")
print(OUTPUT_DIR)
print()


if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Script not found:\n{SCRIPT}"
    )

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"Stage-2 tagged directory not found:\n{INPUT_DIR}"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# IMPORTANT: verify that the NEW Stage-3 script is installed
# ============================================================

version_cmd = [
    "conda",
    "run",
    "--no-capture-output",
    "-n",
    "vdeidspacy",
    "python",
    str(SCRIPT),
    "--version",
]

version_result = subprocess.run(
    version_cmd,
    cwd=str(PROJECT_ROOT),
    check=True,
    capture_output=True,
    text=True,
)

script_version = version_result.stdout.strip()

print("Stage-3 script version:")
print(script_version)
print()

EXPECTED_VERSION = "2026-08-31-stage3-persondata-v4"

if script_version != EXPECTED_VERSION:
    raise RuntimeError(
        "\nWrong/old Stage-3 script detected.\n"
        f"Expected: {EXPECTED_VERSION}\n"
        f"Found:    {script_version}\n\n"
        f"Replace this file:\n{SCRIPT}"
    )


# ============================================================
# Run Stage 3 in vdeidspacy
# ============================================================

cmd = [
    "conda",
    "run",
    "--no-capture-output",
    "-n",
    "vdeidspacy",
    "python",
    str(SCRIPT),

    "--input-dir",
    str(INPUT_DIR),

    "--output-dir",
    str(OUTPUT_DIR),

    "--extensions",
    ".txt",
    ".alfrttm",

    "--batch-size",
    "8",

    # ========================================================
    # Unified annotation format
    #
    # [PersonData]Peter Jensen[PER]
    # [PersonData]København[GPE]
    # ========================================================
    "--annotation-marker",
    "PersonData",

    # ========================================================
    # Cascade rule
    #
    # Existing Stage-2 annotations:
    #
    # [PersonData]Peter Jensen[PER]
    # [PersonData]København[LOC]
    #
    # are protected and NOT sent to DaCy.
    #
    # MISC is the exception:
    #
    # [PersonData]Rigshospitalet[MISC]
    #
    # -> "Rigshospitalet" is reprocessed by DaCy.
    # ========================================================
    "--reprocess-labels",
    "MISC",

    # ========================================================
    # ALFRRTM safety
    #
    # For:
    #
    # start=0.1s stop=3.2s speaker_CLINICIAN "actual text"
    #
    # DaCy receives ONLY:
    #
    # actual text
    #
    # start/stop/speaker metadata is never model input.
    # ========================================================
    "--strict-quoted-utterance",

    # Produce Stage-3 annotation copies
    "--write-tagged-files",
]


print("Running:")
print(" ".join(cmd))
print()


subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    check=True,
)


print("\nDaCy Stage-3 PII recognition finished.")

# DeepLearning
this is finetuning model for better performance

## simple imple xlm-roberta

In [0]:
# cmd run
python pseudopipe/fewshot_pseudonymization_loo_fewnerd_v3.py --input-dir "G1AnonyData_better/G1" --output-dir "out/fewshot_loo_hf_v3/xlm-roberta-large" --extensions .txt .alfrttm --encoder-model "FacebookAI/xlm-roberta-large" --hidden-representation last --model-dtype auto --max-length 128 --encoder-batch-size 8 --query-chunk 64 --distance-budget-mb 256 --structshot-tau 0.05 --tau-sweep "0.001,0.005,0.01,0.05,0.1,0.32" --device auto --seed 42

In [0]:
# =============================================================================
# Jupyter cell: native-HuggingFace leave-one-file-out few-shot pseudonymization
# validation (ProtoBERT + StructShot), using the local uv venv `vdeidpresidio`.
#
# Connected helpers:
#   pseudopipe/config_pipe.py -> HG_TOKEN for authenticated Hugging Face Hub access
#   pseudopipe/loger.py       -> append-mode file + console trace logging
# =============================================================================

import importlib.util
import os
from pathlib import Path
import subprocess

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
SCRIPT = "pseudopipe/fewshot_pseudonymization_loo_fewnerd_v3.py"

# STRICTLY retained from the adapted code.
INPUT_DIR = "G1AnonyData_better/G1"
OUTPUT_ROOT = "out/fewshot_loo_hf_v3"

# Helper modules live in the same pseudopipe/ folder as the main script.
CONFIG_PIPE = "pseudopipe/config_pipe.py"
LOGGER_MODULE = "pseudopipe/loger.py"

# Optional source-domain corpus used only for StructShot transition estimates.
TRANSITION_CORPUS_DIR = None

# Local uv environment. On macOS uv venvs use bin/python.
VENV_DIR = "vdeidpresidio"
PYTHON = str(Path(VENV_DIR) / "bin" / "python")

MAX_LENGTH = 128
ENCODER_BATCH_SIZE = 8
QUERY_CHUNK = 64
DISTANCE_BUDGET_MB = 256
DEVICE = "auto"  # CUDA -> MPS -> CPU
HIDDEN_REPRESENTATION = "last"       # last | mean-last4 | sum-last4
MODEL_DTYPE = "auto"                 # auto | float32 | float16 | bfloat16
NORMALIZE_EMBEDDINGS = True

# Use the dictionary key here, not the raw HF model id.
ENCODER = "xlm-roberta-large"

ENCODERS = {
    "xlm-roberta-large": "FacebookAI/xlm-roberta-large",
    # "danish-xlmr-ner-large": "thomasbeste/danish-xlmr-ner-large",
    # "dacy-large-encoder": "KennethEnevoldsen/dacy-large-encoder",
    # "danish-bert-botxo": "Maltehb/danish-bert-botxo",
    # "danbert-small": "alexanderfalk/danbert-small-cased",
}

RUN = True


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _load_module(path, module_name):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Required helper file not found: {path}")
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import helper module: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def build_child_env():
    """Seed the child environment from config_pipe.py without printing HG_TOKEN."""
    env = os.environ.copy()
    config = _load_module(CONFIG_PIPE, "fewshot_caller_config_pipe")
    token = getattr(config, "HG_TOKEN", None)
    if token is not None:
        token = str(token).strip()
    if token and token.upper() != "YOUR_TOKEN":
        env["HF_TOKEN"] = token
        env["HUGGINGFACE_HUB_TOKEN"] = token
        print(f"Hugging Face authentication: configured from {CONFIG_PIPE}")
    else:
        print(f"Hugging Face authentication: HG_TOKEN is not configured in {CONFIG_PIPE}")
    return env


# ------------------------------------------------------------
# Interpreter / helper validation
# ------------------------------------------------------------
if not Path(PYTHON).is_file():
    raise RuntimeError(
        f"uv environment interpreter not found: {PYTHON}\n"
        "Create it from the project root, e.g.:\n"
        "  uv venv vdeidpresidio\n"
        "  source vdeidpresidio/bin/activate\n"
        "  uv pip install torch transformers nervaluate"
    )

for required in (SCRIPT, CONFIG_PIPE, LOGGER_MODULE):
    if not Path(required).is_file():
        raise FileNotFoundError(f"Required file not found: {Path(required).resolve()}")

print(f"Interpreter: {Path(PYTHON).resolve()}")
child_env = build_child_env()


# ------------------------------------------------------------
# Run one encoder
# ------------------------------------------------------------
def run_encoder(encoder_key):
    if encoder_key not in ENCODERS:
        raise KeyError(
            f"unknown encoder {encoder_key!r}; available: {sorted(ENCODERS)}"
        )

    model = ENCODERS[encoder_key]
    out_dir = os.path.join(OUTPUT_ROOT, encoder_key)
    log_file = os.path.join(out_dir, "fewshot_pseudonymization.log")

    print()
    print("=" * 78)
    print(f"Encoder: {encoder_key} -> {model}")
    print(f"Output : {out_dir}")
    print(f"Log    : {log_file}")
    print("=" * 78)

    cmd = [
        PYTHON,
        SCRIPT,
        "--input-dir", INPUT_DIR,
        "--output-dir", out_dir,
        "--extensions", ".txt", ".alfrttm",
        "--encoder-model", model,
        "--hidden-representation", HIDDEN_REPRESENTATION,
        "--model-dtype", MODEL_DTYPE,
        "--max-length", str(MAX_LENGTH),
        "--encoder-batch-size", str(ENCODER_BATCH_SIZE),
        "--query-chunk", str(QUERY_CHUNK),
        "--distance-budget-mb", str(DISTANCE_BUDGET_MB),
        "--structshot-tau", "0.05",
        "--tau-sweep", "0.001,0.005,0.01,0.05,0.1,0.32",
        "--device", DEVICE,
        "--seed", "42",
        "--config-pipe", CONFIG_PIPE,
        "--logger-module", LOGGER_MODULE,
        "--log-file", log_file,
    ]

    if not NORMALIZE_EMBEDDINGS:
        cmd += ["--no-normalize-embeddings"]

    if TRANSITION_CORPUS_DIR is not None:
        cmd += [
            "--transition-source", "corpus",
            "--transition-corpus-dir", TRANSITION_CORPUS_DIR,
        ]

    print("Running:")
    print(" ".join(cmd))
    print()

    if not RUN:
        print("RUN is False: command not executed.")
        return out_dir

    subprocess.run(cmd, cwd=os.getcwd(), env=child_env, check=True)
    print(f"\nFew-shot HF evaluation finished for {encoder_key}.")
    print("  primary table :", os.path.join(out_dir, "average_metrics.csv"))
    print("  tau sweep     :", os.path.join(out_dir, "structshot_tau_sweep_average_metrics.csv"))
    print("  transition QC :", os.path.join(out_dir, "structshot_transition_audit.csv"))
    print("  implementation:", os.path.join(out_dir, "run_summary.json"))
    print("  trace log     :", log_file)
    return out_dir


# ------------------------------------------------------------
# Go
# ------------------------------------------------------------
encoders_to_run = [ENCODER] if isinstance(ENCODER, str) else list(ENCODER)
written = [run_encoder(key) for key in encoders_to_run]

if RUN and len(written) > 1:
    print()
    print("Compare the encoders with:")
    for d in written:
        print("   ", os.path.join(d, "average_metrics.csv"))


## NOUSE below

## NOUSE da_dacy_large_ner_fine_grained
dacy fine grained model does not have public weights. Not suitable

In [0]:
# Paste this cell into the BASE-conda Jupyter notebook, or run:
#     %run run_fewshot_from_base.py
#
# Single-directory LOO design:
#   - no GOLD_DIR
#   - no VALIDATION_DIR
#   - no separate annotated/source directory
#
# INPUT_DIR contains the already-Presidio-preprocessed sessions used for LOO.
# The same files must contain inline child annotations such as
# [PersonData]Peter[PER]. The validator strips the wrappers before passing text
# to DaCy, so the model sees annotation-free text.

from pathlib import Path
import shutil
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

SCRIPT_CANDIDATES = [
    PROJECT_ROOT / "pseudopipe" / "fewshot_pseudonymization_loo_v3.py",
    PROJECT_ROOT / "fewshot_pseudonymization_loo_v3.py",
]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.is_file()), None)

# This is the ONLY data directory supplied to the validator.
# It should contain the Presidio-preprocessed + child-labelled files.
INPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "preproc_parent_aligned"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "annonydata"
    / "fewshot_pseudonymization_validation"
)

KEEP_FILE_SUBSTRINGS = [#"94", 
                        "95", 
                        #"105", 
                        "169", 
                        "92"]
EXPECTED_FILES = len(KEEP_FILE_SUBSTRINGS) if KEEP_FILE_SUBSTRINGS else 0

CONDA_ENV = "vdeidtorch"
MODEL = "multilingual-e5-large"

# CONDA_ENV = "vdeidspacy"
# MODEL = "da_dacy_large_ner_fine_grained"

SEED = 42
BATCH_SIZE = 8
GPU_ID = -2
STRUCTSHOT_TAU = 0.32
TRANSITION_SMOOTHING = 1.0
MAX_O_SUPPORT = 2048
NORMALIZE_EMBEDDINGS = False
PREFLIGHT_ONLY = False

# Optional JSON mapping overrides. Normally leave as None.
ANNOTATION_LABEL_MAP_JSON = None
DACY_LABEL_MAP_JSON = None

if SCRIPT is None:
    raise FileNotFoundError(
        "Could not find the updated fewshot_pseudonymization_loo.py. "
        "Put it in PROJECT_ROOT/pseudopipe/ or PROJECT_ROOT/."
    )
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")
if shutil.which("conda") is None:
    raise RuntimeError(
        "The 'conda' executable is not available to this Jupyter kernel."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "conda", "run", "--no-capture-output", "-n", CONDA_ENV,
    "python", str(SCRIPT),
    "--input-dir", str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--model", MODEL,
    "--extensions", ".txt", ".alfrttm",
    "--expected-files", str(EXPECTED_FILES),
    "--annotation-marker", "PersonData",
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--gpu-id", str(GPU_ID),
    "--structshot-tau", str(STRUCTSHOT_TAU),
    "--transition-smoothing", str(TRANSITION_SMOOTHING),
    "--max-o-support", str(MAX_O_SUPPORT),
]

if KEEP_FILE_SUBSTRINGS:
    cmd.extend(["--include-substrings", *KEEP_FILE_SUBSTRINGS])
if NORMALIZE_EMBEDDINGS:
    cmd.append("--normalize-embeddings")
if ANNOTATION_LABEL_MAP_JSON is not None:
    cmd.extend([
        "--annotation-label-map-json",
        str(Path(ANNOTATION_LABEL_MAP_JSON)),
    ])
if DACY_LABEL_MAP_JSON is not None:
    cmd.extend([
        "--dacy-label-map-json",
        str(Path(DACY_LABEL_MAP_JSON)),
    ])
if PREFLIGHT_ONLY:
    cmd.append("--preflight-only")

print("Launching leave-one-file-out validation in:", CONDA_ENV)
print("Script:", SCRIPT)
print("Single LOO input directory:", INPUT_DIR)
print("Output:", OUTPUT_DIR)
print("Expected files/folds:", EXPECTED_FILES)

subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

schema_path = OUTPUT_DIR / "gdpr_alf_target_schema.csv"
if schema_path.is_file():
    print("\nGDPR-ALF schema:")
    display(pd.read_csv(schema_path))

if PREFLIGHT_ONLY:
    summary_path = OUTPUT_DIR / "input_annotation_summary.csv"
    if not summary_path.is_file():
        raise FileNotFoundError(
            f"Preflight finished but expected output is missing: {summary_path}"
        )
    print("\nInput annotation summary:")
    display(pd.read_csv(summary_path))
else:
    average_path = OUTPUT_DIR / "average_metrics.csv"
    label_path = OUTPUT_DIR / "average_label_metrics.csv"

    if not average_path.is_file():
        raise FileNotFoundError(
            f"Validation finished but expected output is missing: {average_path}"
        )

    print(f"\nFinal {EXPECTED_FILES}-fold average metrics:")
    display(pd.read_csv(average_path))

    if label_path.is_file():
        label_df = pd.read_csv(label_path)
        print("\nAverage GDPR-ALF child-label metrics:")
        display(
            label_df[label_df["scope"].eq("all_schema")]
            .reset_index(drop=True)
        )

    print("\nDetailed outputs are in:", OUTPUT_DIR)

# post proc

## label align
danish mapping, the parent is due to the initial idea. Keep the fine-grained labels help the LLM training.

In [0]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "pseudopipe/postproc_align_parent_labels.py",
        "--input-dir",
        "annonydata/presidio_deidentified/deidentified",
        "--output-dir",
        # "annonydata/parent_aligned_after_pii", # for the base pipe
        "annonydata/preproc_parent_aligned",
    ],
    check=True,
)

## name recheck

In [0]:
from pathlib import Path
import os
import re
import csv


# ============================================================
# Configuration
# ============================================================

INPUT_DIR = Path("annonydata/parent_aligned_after_pii")
OUTPUT_DIR = Path("annonydata/parent_aligned_per_recheck")

# CSV audit of NEWLY ADDED PER annotations
CSV_PATH = OUTPUT_DIR / "per_recheck_additions.csv"

# CSV audit of seed names that look suspicious (not real names)
# and were therefore EXCLUDED from being propagated as new tags.
# Review this file by hand and fix the original [PER] tags in the
# source data if any of these are genuinely wrong.
FLAGGED_CSV_PATH = OUTPUT_DIR / "per_recheck_flagged_names.csv"

EXTENSIONS = {".alfrttm", ".txt"}

EXCLUDED_DIRS = {
    "__pycache__",
    "node_modules",
    "System Volume Information",
    "$RECYCLE.BIN",
}


# ============================================================
# Non-name dictionary
# ============================================================
#
# Words/tokens that have shown up as [PER] seeds in the past but are
# NOT real human names (common words, numbers, single letters, stray
# tokens, etc.). Anything whose casefolded text appears here is:
#
#   1. NOT used as a seed to propagate new [PersonData]...[PER] tags
#      elsewhere in the file (so we don't spam false positives), and
#   2. Written to FLAGGED_CSV_PATH so a human can inspect the
#      *original* annotation and decide whether to fix/remove it.
#
# The existing tag in the source text is left untouched either way -
# this dictionary only controls whether the seed is allowed to spread.
#
# Extend this set as you discover more bad seeds during review.
NON_NAME_DICTIONARY = {
    # generic/greeting words that sometimes get capitalized and
    # mis-tagged
    "ok", "okay", "hej", "hejsa", "goddag", "farvel",
    "tak", "tak for det", "please", "hello", "hi", "yes", "no",
    "ja", "nej", "jo",
    # days / months (capitalized -> can look like names)
    "mandag", "tirsdag", "onsdag", "torsdag", "fredag",
    "lordag", "lørdag", "sondag", "søndag",
    "januar", "februar", "marts", "april", "maj", "juni",
    "juli", "august", "september", "oktober", "november",
    "december",
    # generic titles / pronouns that sometimes slip through
    "mor", "far", "mormor", "farmor", "morfar", "farfar",
    "han", "hun", "de", "det", "den",
    # placeholders / system-ish tokens
    "unknown", "n/a", "na", "test", "none", "null",
}


def is_non_name(candidate):
    """True if candidate looks like it is NOT a real human name."""
    key = candidate.strip().casefold()

    if not key:
        return True

    if key in NON_NAME_DICTIONARY:
        return True

    # Pure digits ("105") or a single character are almost never
    # real names.
    if key.isdigit():
        return True

    if len(key) <= 1:
        return True

    return False


# ============================================================
# Annotation regex
# ============================================================

# Protect any existing annotation, e.g.:
#
#   [PersonData]Peter Jensen[PER]
#   [PersonData]105[MISC]
#   [PersonData]København[LOC]
#
# NOTE: the name body and the label both need + / * quantifiers.
# Without them this regex only matches a single-character name and a
# two-character label, so it fails to match real annotations like
# [PER] (3 chars) or [MISC] (4 chars) - which was the root cause of
# annotations getting doubled up ([PersonData][PersonData]X[PER][PER]).
ANNOTATION_RE = re.compile(
    r"\[PersonData\][^\[\]\r\n]+\[[A-Z][A-Z0-9_]*\]"
)

# Collect ONLY exact uppercase [PER] annotations.
PER_RE = re.compile(
    r"\[PersonData\]([^\[\]\r\n]+?)\[PER\]"
)


# ============================================================
# Input-file discovery
# ============================================================

def is_excluded_dir(dirname):
    return (
        dirname.startswith(".")
        or dirname in EXCLUDED_DIRS
    )


def find_input_files(input_dir):
    input_dir = Path(input_dir).resolve()
    output_dir = OUTPUT_DIR.resolve()

    files = []

    for root, dirs, filenames in os.walk(input_dir):
        root_path = Path(root)

        # Skip hidden/protected folders
        dirs[:] = [
            d for d in dirs
            if not is_excluded_dir(d)
        ]

        # Avoid reading our own output if OUTPUT_DIR is under INPUT_DIR
        dirs[:] = [
            d for d in dirs
            if (root_path / d).resolve() != output_dir
        ]

        for filename in filenames:

            if filename.startswith("."):
                continue

            path = root_path / filename

            if path.suffix.lower() in EXTENSIONS:
                files.append(path)

    return sorted(files)


# ============================================================
# Collect existing PER names
# ============================================================

def collect_per_names(text):
    """
    Collect names only from:

        [PersonData]XX[PER]

    [PER] must be uppercase.

    Case variants are treated as the same seed:
        Peter
        peter
        PETER

    Returns:
        names          -> accepted seeds, longest first, used to
                           propagate new tags
        flagged_names  -> seeds that matched NON_NAME_DICTIONARY (or
                           otherwise look non-name-like); NOT used to
                           propagate, only logged for human review
    """

    names_by_casefold = {}

    for match in PER_RE.finditer(text):

        name = match.group(1).strip()

        if not name:
            continue

        key = name.casefold()

        if key not in names_by_casefold:
            names_by_casefold[key] = name

    accepted = []
    flagged = []

    for name in names_by_casefold.values():
        if is_non_name(name):
            flagged.append(name)
        else:
            accepted.append(name)

    # Longest first:
    # Peter Jensen before Peter
    accepted.sort(key=len, reverse=True)

    return accepted, flagged


def build_name_pattern(names):
    if not names:
        return None

    alternatives = "|".join(
        re.escape(name)
        for name in names
    )

    return re.compile(
        rf"(?<!\w)(?:{alternatives})(?!\w)",
        flags=re.IGNORECASE,
    )


# ============================================================
# Helper functions for CSV information
# ============================================================

def get_line_number(text, absolute_position):
    """1-based line number for an absolute character position."""
    return text.count("\n", 0, absolute_position) + 1


def get_start_in_line(text, absolute_position):
    """0-based character offset within the line."""
    previous_newline = text.rfind("\n", 0, absolute_position)

    if previous_newline == -1:
        return absolute_position

    return absolute_position - previous_newline - 1


# ============================================================
# Tag unannotated repetitions + record additions
# ============================================================

def tag_unannotated_text(text, name_pattern, names):
    """
    Search only outside existing PersonData annotations.

    Returns:
        new_text
        list of audit records
    """

    if name_pattern is None:
        return text, []

    # Maps:
    #   "peter jensen" -> "Peter Jensen"
    #
    # so a newly found PETER JENSEN can be linked back to
    # the existing PER seed.
    seed_lookup = {
        name.casefold(): name
        for name in names
    }

    output = []
    last_end = 0
    additions = []

    def process_plain_segment(plain, absolute_segment_start):

        segment_output = []
        previous = 0

        for match in name_pattern.finditer(plain):

            found_text = match.group(0)

            absolute_start = (
                absolute_segment_start
                + match.start()
            )

            absolute_end = (
                absolute_segment_start
                + match.end()
            )

            annotation = (
                f"[PersonData]{found_text}[PER]"
            )

            # Text before match
            segment_output.append(
                plain[previous:match.start()]
            )

            # New annotation
            segment_output.append(annotation)

            canonical_seed = seed_lookup.get(
                found_text.casefold(),
                found_text,
            )

            additions.append({
                "line_number": get_line_number(
                    text,
                    absolute_start,
                ),
                "seed_name": canonical_seed,
                "found_name": found_text,
                "start_in_line": get_start_in_line(
                    text,
                    absolute_start,
                ),
                "end_in_line": get_start_in_line(
                    text,
                    absolute_end,
                ),
                "start_in_file": absolute_start,
                "end_in_file": absolute_end,
                "annotation": annotation,
            })

            previous = match.end()

        segment_output.append(
            plain[previous:]
        )

        return "".join(segment_output)

    # --------------------------------------------------------
    # Only process text BETWEEN existing annotations.
    # Existing annotations themselves are copied unchanged.
    # --------------------------------------------------------

    for annotation in ANNOTATION_RE.finditer(text):

        plain = text[
            last_end:annotation.start()
        ]

        output.append(
            process_plain_segment(
                plain,
                last_end,
            )
        )

        # Preserve existing annotation EXACTLY
        output.append(
            annotation.group(0)
        )

        last_end = annotation.end()

    # Text after final annotation
    plain = text[last_end:]

    output.append(
        process_plain_segment(
            plain,
            last_end,
        )
    )

    return "".join(output), additions


# ============================================================
# File-level recheck
# ============================================================

def recheck_per_file(text):

    names, flagged_names = collect_per_names(text)

    if not names:
        return text, names, flagged_names, []

    name_pattern = build_name_pattern(names)

    new_text, additions = tag_unannotated_text(
        text,
        name_pattern,
        names,
    )

    return new_text, names, flagged_names, additions


# ============================================================
# Run
# ============================================================

INPUT_DIR = INPUT_DIR.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()
CSV_PATH = OUTPUT_DIR / "per_recheck_additions.csv"
FLAGGED_CSV_PATH = OUTPUT_DIR / "per_recheck_flagged_names.csv"

if not INPUT_DIR.is_dir():
    raise NotADirectoryError(
        f"Input directory does not exist: {INPUT_DIR}"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

files = find_input_files(INPUT_DIR)

print(f"Found {len(files)} eligible files")
print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}\n")


all_additions = []
all_flagged = []

total_added = 0
files_with_additions = 0


for src in files:

    relative = src.relative_to(INPUT_DIR)
    dst = OUTPUT_DIR / relative

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    text = src.read_text(
        encoding="utf-8",
        errors="replace",
    )

    new_text, per_names, flagged_names, additions = recheck_per_file(
        text
    )

    dst.write_text(
        new_text,
        encoding="utf-8",
    )

    # Add filename information to each CSV record
    for record in additions:
        record["file"] = str(relative)

    all_additions.extend(additions)

    for flagged_name in flagged_names:
        all_flagged.append({
            "file": str(relative),
            "seed_name": flagged_name,
            "reason": "matched non-name dictionary / heuristic",
        })

    n_added = len(additions)
    total_added += n_added

    if n_added > 0:
        files_with_additions += 1

    print(
        f"{relative}: "
        f"{len(per_names)} accepted PER seed(s), "
        f"{len(flagged_names)} flagged seed(s), "
        f"{n_added} new occurrence(s)"
    )


# ============================================================
# Write CSV audit - new annotations
# ============================================================

CSV_FIELDS = [
    "file",
    "line_number",
    "seed_name",
    "found_name",
    "start_in_line",
    "end_in_line",
    "start_in_file",
    "end_in_file",
    "annotation",
]

with CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=CSV_FIELDS,
    )

    writer.writeheader()

    for record in all_additions:
        writer.writerow(record)


# ============================================================
# Write CSV audit - flagged (non-name) seeds for human review
# ============================================================

FLAGGED_CSV_FIELDS = ["file", "seed_name", "reason"]

with FLAGGED_CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=FLAGGED_CSV_FIELDS,
    )

    writer.writeheader()

    for record in all_flagged:
        writer.writerow(record)


# ============================================================
# Summary
# ============================================================

print("\nDone.")
print(f"Files processed: {len(files)}")
print(f"Files with additions: {files_with_additions}")
print(f"Total new PER annotations: {total_added}")
print(f"Total flagged (non-name) seeds: {len(all_flagged)}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"CSV audit (additions): {CSV_PATH}")
print(f"CSV audit (flagged seeds for review): {FLAGGED_CSV_PATH}")

## deidentify

In [0]:
from pathlib import Path
from collections import Counter
import os
import re
import random


# ============================================================
# Paths
# ============================================================

INPUT_DIR = Path(r"annonydata\parent_aligned_per_recheck")
OUTPUT_DIR = Path(r"annonydata\pii_replaced")

EXTENSIONS = {".alfrttm", ".txt"}

# Reproducible pseudonymization.
# Change the number if you want another random realization.
SEED = 42
rng = random.Random(SEED)


# ============================================================
# Input label -> Danish target label
# ============================================================

LABEL_TO_TARGET = {
    # PER
    "PER": "PERSON",
    "STAFF": "PERSONALE",
    "PATIENT": "PATIENT",
    "PERSON": "PERSON",

    # LOC
    "LOC": "STED",
    "ADDRESS": "ADRESSE",
    "POSTCODE": "POSTNUMMER",
    "GPE": "GEOPOLITISK_ENHED",
    "LOCATION": "STED",
    "FACILITY": "FACILITET",

    # ORG
    "ORG": "ORGANISATION",
    "HOSPITAL": "HOSPITAL",
    "ORGANIZATION": "ORGANISATION",

    # CONTACT
    "CONTACT": "KONTAKT",
    "PHONE": "TELEFON",
    "EMAIL": "EMAIL",

    # ID
    "ID": "IDENTIFIKATOR",
    "CPR": "CPR",
    "IBAN": "IBAN",
    "IP": "IP",
    "URL": "URL",

    # DATE
    "DATE": "DATO",
    "TIME": "TID",
    "DURATION": "VARIGHED",

    # DEM
    "DEM": "DEMOGRAFI",
    "ETHNICITY": "ETNICITET",
    "RELIGION": "RELIGION",
    "POLITICS": "POLITIK",
    "SEXUALITY": "SEKSUALITET",
    "AGE": "ALDER",
    "LANGUAGE": "SPROG",
    "NORP": "GRUPPETILHØRSFORHOLD",

    # HEALTH
    "HEALTH": "HELBRED",
    "DIAGNOSIS": "DIAGNOSE",
    "MEDICATION": "MEDICIN",
    "CONDITION": "TILSTAND",

    # Numeric
    "CARDINAL": "TAL",
    "MONEY": "BELØB",
    "ORDINAL": "TAL",
    "PERCENT": "PROCENT",
    "QUANTITY": "TAL",

    # Misc
    "MISC": "PERSONOPLYSNING",
    "EVENT": "BEGIVENHED",
    "LAW": "LOV",
    "PRODUCT": "PRODUKT",
    "WORK_OF_ART": "VÆRK",
}


# Also allow files which already contain the Danish target labels.
TARGET_LABELS = set(LABEL_TO_TARGET.values())


# ============================================================
# Replacement material from the PDF
# ============================================================

NAMES = [
    "Anne", "Mette", "Kirsten", "Hanne", "Anna",
    "Peter", "Michael", "Lars", "Thomas", "Jens",
]

DANISH_CITIES = [
    "København",
    "Aarhus",
    "Odense",
    "Aalborg",
    "Esbjerg",
    "Roskilde",
    "Vejle",
    "Horsens",
]

# Synthetic Danish-looking addresses.
STREETS = [
    "Birkevej",
    "Skovvej",
    "Parkvej",
    "Engvej",
    "Søvej",
    "Bakkevej",
]

ORG_TYPES = [
    "skole",
    "universitet",
    "gymnasium",
    "hospital",
    "virksomhed",
    "organisation",
    "myndighed",
    "kommune",
    "forening",
    "institution",
]

DANISH_HOSPITALS = [
    "Rigshospitalet",
    "Bispebjerg Hospital",
    "Herlev Hospital",
    "Hvidovre Hospital",
    "Odense Universitetshospital",
]

ETHNICITIES = [
    "dansk",
    "tysk",
    "svensk",
    "norsk",
    "polsk",
    "tyrkisk",
    "somalisk",
    "arabisk",
    "pakistansk",
    "kinesisk",
]

RELIGIONS = [
    "kristendom",
    "islam",
    "hinduisme",
]

POLITICS = [
    "venstre",
    "højre",
    "republikaner",
    "liberal",
]

SEXUALITIES = [
    "heteroseksuel",
    "homoseksuel",
    "biseksuel",
    "panseksuel",
    "aseksuel",
]

CONDITIONS = [
    "diabetes",
    "kræft",
    "depression",
    "hjerte-kar-sygdom",
    "astma",
]


# ============================================================
# Replacement generators
# ============================================================

def random_digits(n):
    return "".join(
        str(rng.randint(0, 9))
        for _ in range(n)
    )


def random_phone():
    return "+45 " + random_digits(8)


def random_email():
    return f"{rng.choice(NAMES)}@gmail.com"


def random_postcode():
    # PDF: random 2-digit number + 00
    return f"{rng.randint(10, 99)}00"


def random_date():
    # PDF: day 1-30 / month 1-12 / year 2000-2026
    return (
        f"{rng.randint(1, 30)}/"
        f"{rng.randint(1, 12)}/"
        f"{rng.randint(2000, 2026)}"
    )


def random_time():
    # Follow PDF literally: 1-24 / 1-59
    return (
        f"{rng.randint(1, 24):02d}:"
        f"{rng.randint(1, 59):02d}"
    )


def random_age():
    # The PDF explicitly shows the first blocks:
    # 18-26, 27-35, ...
    #
    # Restricting this implementation to those explicitly
    # shown blocks rather than inventing later boundaries.
    low, high = rng.choice([
        (18, 26),
        (27, 35),
    ])

    return str(rng.randint(low, high))


# ============================================================
# Replacement policy
# ============================================================

def replacement_for(target_label):
    """
    Return replacement text.

    If the PDF provides no usable replacement solution,
    return [TARGET_LABEL].

    MISC/PERSONOPLYSNING is therefore always:
        [PERSONOPLYSNING]
    """

    # ---------- PER ----------

    if target_label == "PERSON":
        return rng.choice(NAMES)

    if target_label == "PERSONALE":
        return "Psychologist"

    if target_label == "PATIENT":
        return rng.choice(NAMES)

    # ---------- LOC ----------

    if target_label == "STED":
        return rng.choice(DANISH_CITIES)

    if target_label == "ADRESSE":
        return (
            f"{rng.choice(STREETS)} "
            f"{rng.randint(1, 199)}"
        )

    if target_label == "POSTNUMMER":
        return random_postcode()

    # GEOPOLITISK_ENHED and FACILITET have no
    # explicit replacement rule in the PDF.

    # ---------- ORG ----------

    if target_label == "ORGANISATION":
        return rng.choice(ORG_TYPES)

    if target_label == "HOSPITAL":
        return rng.choice(DANISH_HOSPITALS)

    # ---------- CONTACT ----------

    if target_label == "KONTAKT":
        # PDF: ambiguous OR +45 + random 8 digits.
        # Use its explicit phone option.
        return random_phone()

    if target_label == "TELEFON":
        return random_phone()

    if target_label == "EMAIL":
        return random_email()

    # ---------- ID ----------

    if target_label == "IDENTIFIKATOR":
        return random_digits(10)

    if target_label == "CPR":
        return random_digits(10)

    # IBAN, IP and URL have no replacement rule in this PDF.

    # ---------- DATE ----------

    if target_label == "DATO":
        return random_date()

    if target_label == "TID":
        return random_time()

    if target_label == "VARIGHED":
        return random_time()

    # ---------- DEM ----------

    # DEMOGRAFI is "ambiguous" in the PDF -> placeholder.

    if target_label == "ETNICITET":
        return rng.choice(ETHNICITIES)

    if target_label == "RELIGION":
        return rng.choice(RELIGIONS)

    if target_label == "POLITIK":
        return rng.choice(POLITICS)

    if target_label == "SEKSUALITET":
        return rng.choice(SEXUALITIES)

    if target_label == "ALDER":
        return random_age()

    # SPROG / GRUPPETILHØRSFORHOLD are not specified.

    # ---------- HEALTH ----------

    # HELBRED is "ambiguous" -> placeholder.

    if target_label == "DIAGNOSE":
        return "schizofrenispektrumdiagnose"

    if target_label == "MEDICIN":
        return "antipsykotisk medikation"

    if target_label == "TILSTAND":
        return rng.choice(CONDITIONS)

    # ---------- MISC ----------

    if target_label == "PERSONOPLYSNING":
        return "[PERSONOPLYSNING]"

    # --------------------------------------------------------
    # No solution specified in PDF:
    #
    # [IBAN]
    # [IP]
    # [URL]
    # [FACILITET]
    # [GEOPOLITISK_ENHED]
    # [DEMOGRAFI]
    # [SPROG]
    # [HELBRED]
    # [TAL]
    # [BELØB]
    # [PROCENT]
    # [BEGIVENHED]
    # etc.
    # --------------------------------------------------------

    return f"[{target_label}]"


# ============================================================
# Annotation matching
# ============================================================

# Example:
#
#   [PersonData]Peter Jensen[PER]
#   [PersonData]105[MISC]
#   [PersonData]København[LOC]
#
ANNOTATION_RE = re.compile(
    r"\[PersonData\]"
    r"(?P<text>[^\[\]\r\n]*?)"
    r"\[(?P<label>[A-ZÆØÅ][A-ZÆØÅ0-9_]*)\]"
)


def replace_annotations(text, counter):
    """
    Replace complete [PersonData]text[LABEL] annotation with
    pseudonymized text.

    The PersonData wrapper and original label disappear.
    """

    def repl(match):
        input_label = match.group("label")

        # If source label -> map to Danish target label.
        if input_label in LABEL_TO_TARGET:
            target_label = LABEL_TO_TARGET[input_label]

        # If input has already been mapped, use it directly.
        elif input_label in TARGET_LABELS:
            target_label = input_label

        # Completely unknown label:
        # preserve privacy using its own placeholder.
        else:
            target_label = input_label

        replacement = replacement_for(target_label)

        counter[(input_label, target_label)] += 1

        return replacement

    return ANNOTATION_RE.sub(repl, text)


# ============================================================
# File discovery
# ============================================================

EXCLUDED_DIRS = {
    "__pycache__",
    "node_modules",
    "$RECYCLE.BIN",
    "System Volume Information",
}


def find_files(input_dir):
    files = []

    for root, dirs, filenames in os.walk(input_dir):

        # Do not enter:
        # .ipynb_checkpoints, .git, .cache, etc.
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
            and d not in EXCLUDED_DIRS
        ]

        for filename in filenames:

            if filename.startswith("."):
                continue

            path = Path(root) / filename

            if path.suffix.lower() in EXTENSIONS:
                files.append(path)

    return sorted(files)


# ============================================================
# Run
# ============================================================

INPUT_DIR = INPUT_DIR.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()

if not INPUT_DIR.is_dir():
    raise NotADirectoryError(
        f"Input directory does not exist: {INPUT_DIR}"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

files = find_files(INPUT_DIR)

print(f"Found {len(files)} eligible files")
print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}\n")

replacement_counts = Counter()


for src in files:

    relative = src.relative_to(INPUT_DIR)

    # No extra "tagged" directory is introduced.
    dst = OUTPUT_DIR / relative

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    text = src.read_text(
        encoding="utf-8",
        errors="replace",
    )

    replaced_text = replace_annotations(
        text,
        replacement_counts,
    )

    dst.write_text(
        replaced_text,
        encoding="utf-8",
    )

    print(f"{relative} -> {dst}")


# ============================================================
# Summary
# ============================================================

print("\nDone.")
print(f"Files processed: {len(files)}")
print("\nReplacement counts:")

for (input_label, target_label), count in sorted(
    replacement_counts.items()
):
    print(
        f"  {input_label:20s} -> "
        f"{target_label:25s}: {count}"
    )

print(f"\nOutput directory: {OUTPUT_DIR}")

# MISC-stas label dist
run this in some sub venvs

In [0]:
# run this in some sub venvs

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# files already exists, e.g.
# files = ["annonydata/a_recheck.alfrttm", "annonydata/b_consensus.alfrttm"]

# Matches: [PersonData]some pseudonymized text[LABEL]
pattern = re.compile(r"\[PersonData\](.*?)\[([^\[\]\r\n]+)\]")

rows = []

for file in files:
    with open(file, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()

    labels = [label.strip() for _, label in pattern.findall(text)]
    counts = Counter(labels)

    print(f"\n{file}")
    print(f"  Total labels: {len(labels)}")
    print(f"  Candidate labels: {sorted(counts)}")

    for label, count in counts.items():
        rows.append({
            "file": os.path.basename(file),
            "label": label,
            "count": count
        })

# Long-format table
df = pd.DataFrame(rows)

# File × label count matrix
matrix = (
    df.pivot_table(
        index="file",
        columns="label",
        values="count",
        fill_value=0
    )
    .astype(int)
)

print("\nLabel distribution:")
print(matrix)

matrix.to_csv("pseudonymization_label_distribution.csv")

# Visualization
ax = matrix.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

ax.set_xlabel("File")
ax.set_ylabel("Number of labels")
ax.set_title("Pseudonymization label distribution by file")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("pseudonymization_label_distribution.png", dpi=300)
plt.show()

# MISC-chk alfrttm

In [0]:
from pathlib import Path
import csv
import re
from dataclasses import dataclass, field
from typing import Optional


# ============================================================
# SETTINGS
# ============================================================

# Use 15.0 or 20.0 depending on your preferred maximum duration.
MAX_DURATION = 20.0

# When True, missing or extreme timestamps may be reconstructed
# from the previous segment's stop and/or the next segment's start.
USE_CONTEXT_INFERENCE = True

# Enter either:
#   1. One .alfrttm file
#   2. A folder containing multiple .alfrttm files
INPUT_TARGET = Path(
    "92_9_truth.alfrttm"
)


# ============================================================
# PARSING RULES
# ============================================================

LINE_PATTERN = re.compile(
    r"""
    ^\s*
    start\s*=\s*(?P<start>.*?)
    \s*(?=(?:stop|end)\s*=)
    (?:stop|end)\s*=\s*(?P<stop>.*?)
    \s*(?=speaker_)
    (?P<speaker>speaker_[A-Za-z0-9_]+)
    \s*(?P<text>.*?)
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE,
)


@dataclass
class Segment:
    line_number: int
    original_line: str

    is_blank: bool = False
    parsed: bool = False

    raw_start: str = ""
    raw_stop: str = ""
    speaker: str = ""
    raw_text: str = ""

    start: Optional[float] = None
    stop: Optional[float] = None
    corrected_text: str = ""

    issues: list[str] = field(default_factory=list)
    actions: list[str] = field(default_factory=list)

    status: str = "ok"
    corrected_line: str = ""


# ============================================================
# NORMALIZATION FUNCTIONS
# ============================================================

def format_timestamp(value: float) -> str:
    """
    Convert a numeric timestamp to canonical ALFRTTM format.

    Examples:
        19     -> 19.0s
        19.5   -> 19.5s
        19.125 -> 19.125s
    """
    formatted = f"{value:.3f}".rstrip("0").rstrip(".")

    if "." not in formatted:
        formatted += ".0"

    return formatted + "s"


def normalize_timestamp(
    raw_timestamp: str,
    field_name: str
) -> tuple[Optional[float], list[str], list[str]]:
    """
    Repair a timestamp where possible.

    Repairs include:
    - Spaces inside the timestamp
    - Missing 's'
    - Comma decimal separators
    - Extra decimal points, such as 1234.5.6s
    - Invalid characters
    - Negative timestamps

    A timestamp with no number cannot be reconstructed here.
    Contextual reconstruction is performed later.
    """
    issues = []
    actions = []

    token = (raw_timestamp or "").strip()

    # Remove spaces: "19 . 0 s" -> "19.0s"
    if re.search(r"\s", token):
        issues.append(f"{field_name}_timestamp_contains_spaces")
        token = re.sub(r"\s+", "", token)
        actions.append(f"removed_spaces_from_{field_name}")

    # Convert decimal commas: "19,5s" -> "19.5s"
    if "," in token:
        issues.append(f"{field_name}_comma_decimal_separator")
        token = token.replace(",", ".")
        actions.append(f"replaced_comma_in_{field_name}")

    # Check the required s suffix.
    if not re.search(r"[sS]$", token):
        issues.append(f"{field_name}_missing_s_suffix")
        actions.append(f"added_s_to_{field_name}")

    # Remove one or more trailing s characters before numeric parsing.
    token = re.sub(r"[sS]+$", "", token)

    # Cases such as start=s
    if token == "" or not re.search(r"\d", token):
        issues.append(f"{field_name}_missing_number")
        return None, issues, actions

    # Remove unexpected characters.
    cleaned_token = re.sub(r"[^0-9.\-+]", "", token)

    if cleaned_token != token:
        issues.append(f"{field_name}_invalid_characters")
        token = cleaned_token
        actions.append(
            f"removed_invalid_characters_from_{field_name}"
        )

    # Repair timestamps such as 1234.5.6s.
    # This retains the first decimal point and removes later ones:
    # 1234.5.6 -> 1234.56
    if token.count(".") > 1:
        issues.append(f"{field_name}_multiple_decimal_points")

        parts = token.split(".")
        token = parts[0] + "." + "".join(parts[1:])

        actions.append(
            f"removed_extra_decimal_points_from_{field_name}"
        )

    try:
        value = float(token)

    except ValueError:
        # Final conservative attempt: reconstruct from numeric groups.
        numeric_groups = re.findall(r"\d+", token)

        if not numeric_groups:
            issues.append(f"{field_name}_unrecoverable")
            return None, issues, actions

        if len(numeric_groups) == 1:
            value = float(numeric_groups[0])
        else:
            value = float(
                numeric_groups[0] + "." + "".join(numeric_groups[1:])
            )

        issues.append(f"{field_name}_malformed")
        actions.append(
            f"reconstructed_{field_name}_from_numeric_groups"
        )

    if value < 0:
        issues.append(f"{field_name}_negative")
        value = abs(value)
        actions.append(f"converted_{field_name}_to_positive")

    return value, issues, actions


def normalize_transcript(
    raw_text: str
) -> tuple[str, list[str], list[str]]:
    """
    Ensure that every transcript has opening and closing quotation marks.

    Examples:
        Ja.     -> "Ja."
        "Ja.    -> "Ja."
        Ja."    -> "Ja."
        empty   -> ""
    """
    issues = []
    actions = []

    text = (raw_text or "").strip()

    has_complete_quotes = (
        len(text) >= 2
        and text.startswith('"')
        and text.endswith('"')
    )

    if not has_complete_quotes:
        issues.append("missing_or_unbalanced_transcript_quotes")
        actions.append("added_transcript_quotes")

    # Remove existing outer quotation marks independently.
    if text.startswith('"'):
        text = text[1:]

    if text.endswith('"'):
        text = text[:-1]

    # Escape quotation marks occurring inside the transcript.
    text = re.sub(r'(?<!\\)"', r'\\"', text)

    return f'"{text}"', issues, actions


# ============================================================
# FILE PARSING
# ============================================================

def parse_segments(lines: list[str]) -> list[Segment]:
    segments = []

    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.rstrip("\r\n")

        segment = Segment(
            line_number=line_number,
            original_line=line
        )

        if not line.strip():
            segment.is_blank = True
            segment.corrected_line = ""
            segments.append(segment)
            continue

        match = LINE_PATTERN.match(line)

        if not match:
            segment.issues.append("unparseable_line")
            segment.status = "manual_review"
            segment.corrected_line = line
            segments.append(segment)
            continue

        segment.parsed = True
        segment.raw_start = match.group("start")
        segment.raw_stop = match.group("stop")
        segment.speaker = match.group("speaker")
        segment.raw_text = match.group("text")

        (
            segment.start,
            timestamp_issues,
            timestamp_actions
        ) = normalize_timestamp(
            segment.raw_start,
            "start"
        )

        segment.issues.extend(timestamp_issues)
        segment.actions.extend(timestamp_actions)

        (
            segment.stop,
            timestamp_issues,
            timestamp_actions
        ) = normalize_timestamp(
            segment.raw_stop,
            "stop"
        )

        segment.issues.extend(timestamp_issues)
        segment.actions.extend(timestamp_actions)

        (
            segment.corrected_text,
            text_issues,
            text_actions
        ) = normalize_transcript(segment.raw_text)

        segment.issues.extend(text_issues)
        segment.actions.extend(text_actions)

        segments.append(segment)

    return segments


# ============================================================
# CONTEXTUAL TIMESTAMP REPAIR
# ============================================================

def find_previous_stop(
    segments: list[Segment],
    current_index: int
) -> Optional[float]:
    for index in range(current_index - 1, -1, -1):
        candidate = segments[index]

        if candidate.parsed and candidate.stop is not None:
            return candidate.stop

    return None


def find_next_start(
    segments: list[Segment],
    current_index: int
) -> Optional[float]:
    for index in range(current_index + 1, len(segments)):
        candidate = segments[index]

        if candidate.parsed and candidate.start is not None:
            return candidate.start

    return None


def repair_segments(
    segments: list[Segment],
    max_duration: float = 20.0,
    use_context_inference: bool = True
) -> list[Segment]:

    for index, segment in enumerate(segments):

        if segment.is_blank or not segment.parsed:
            continue

        previous_stop = find_previous_stop(segments, index)
        next_start = find_next_start(segments, index)

        # ----------------------------------------------------
        # Infer a missing start timestamp
        # ----------------------------------------------------
        if (
            segment.start is None
            and use_context_inference
            and segment.stop is not None
            and previous_stop is not None
            and 0 <= segment.stop - previous_stop <= max_duration
        ):
            segment.start = previous_stop
            segment.actions.append(
                "inferred_start_from_previous_stop"
            )
            segment.status = "heuristic_correction"

        # ----------------------------------------------------
        # Infer a missing stop timestamp
        # ----------------------------------------------------
        if (
            segment.stop is None
            and use_context_inference
            and segment.start is not None
            and next_start is not None
            and 0 <= next_start - segment.start <= max_duration
        ):
            segment.stop = next_start
            segment.actions.append(
                "inferred_stop_from_next_start"
            )
            segment.status = "heuristic_correction"

        # A missing timestamp that cannot be inferred requires review.
        if segment.start is None or segment.stop is None:
            segment.status = "manual_review"
            segment.corrected_line = segment.original_line
            continue

        # ----------------------------------------------------
        # Start must be earlier than stop
        # ----------------------------------------------------
        if segment.start > segment.stop:
            segment.issues.append("start_after_stop")

            segment.start, segment.stop = (
                segment.stop,
                segment.start
            )

            segment.actions.append("swapped_start_and_stop")

        # ----------------------------------------------------
        # Repair zero-duration segments
        # ----------------------------------------------------
        # Example:
        #   start=3398.1s stop=3398.1s speaker_LAUGH ""
        # becomes:
        #   start=3397.8s stop=3398.1s speaker_LAUGH ""
        if segment.start == segment.stop:
            segment.issues.append("zero_duration_segment")
            segment.start = max(0.0, segment.start - 0.3)
            segment.actions.append("moved_zero_duration_start_back_0.3s")
            segment.status = "auto_corrected"

        duration = segment.stop - segment.start

        # ----------------------------------------------------
        # Repair or flag excessively long segments
        # ----------------------------------------------------
        if duration > max_duration:
            segment.issues.append(
                f"segment_longer_than_{max_duration:g}s"
            )

            repaired = False

            if use_context_inference:

                # Use both neighbouring boundaries when they form
                # a plausible short interval.
                if (
                    previous_stop is not None
                    and next_start is not None
                    and 0 <= next_start - previous_stop <= max_duration
                ):
                    segment.start = previous_stop
                    segment.stop = next_start

                    segment.actions.append(
                        "replaced_outlier_times_with_neighbour_boundaries"
                    )

                    segment.status = "heuristic_correction"
                    repaired = True

                # The stop timestamp appears to be the outlier.
                elif (
                    next_start is not None
                    and 0 <= next_start - segment.start <= max_duration
                ):
                    segment.stop = next_start

                    segment.actions.append(
                        "replaced_long_segment_stop_with_next_start"
                    )

                    segment.status = "heuristic_correction"
                    repaired = True

                # The start timestamp appears to be the outlier.
                elif (
                    previous_stop is not None
                    and 0 <= segment.stop - previous_stop <= max_duration
                ):
                    segment.start = previous_stop

                    segment.actions.append(
                        "replaced_long_segment_start_with_previous_stop"
                    )

                    segment.status = "heuristic_correction"
                    repaired = True

            if not repaired:
                # The true timestamps cannot be known safely.
                segment.status = "manual_review"

        if segment.status == "ok" and segment.issues:
            segment.status = "auto_corrected"

        segment.corrected_line = (
            f"start={format_timestamp(segment.start)} "
            f"stop={format_timestamp(segment.stop)} "
            f"{segment.speaker} "
            f"{segment.corrected_text}"
        )

    return segments


# ============================================================
# OUTPUT
# ============================================================

def write_error_report(
    source_file: Path,
    segments: list[Segment]
) -> Path:
    """
    Example:
        session01.alfrttm -> errsession01.csv
    """
    output_csv = source_file.with_name(
        f"err{source_file.stem}.csv"
    )

    fieldnames = [
        "line_number",
        "status",
        "original_start",
        "original_stop",
        "corrected_start",
        "corrected_stop",
        "corrected_duration_seconds",
        "speaker",
        "issues",
        "actions",
        "original_line",
        "corrected_line"
    ]

    with output_csv.open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for segment in segments:
            # Only include lines containing errors or review decisions.
            if not segment.issues and segment.status == "ok":
                continue

            duration = ""

            if (
                segment.start is not None
                and segment.stop is not None
            ):
                duration = round(
                    segment.stop - segment.start,
                    3
                )

            writer.writerow({
                "line_number": segment.line_number,
                "status": segment.status,
                "original_start": segment.raw_start,
                "original_stop": segment.raw_stop,
                "corrected_start": (
                    format_timestamp(segment.start)
                    if segment.start is not None
                    else ""
                ),
                "corrected_stop": (
                    format_timestamp(segment.stop)
                    if segment.stop is not None
                    else ""
                ),
                "corrected_duration_seconds": duration,
                "speaker": segment.speaker,
                "issues": "; ".join(segment.issues),
                "actions": "; ".join(segment.actions),
                "original_line": segment.original_line,
                "corrected_line": segment.corrected_line
            })

    return output_csv


def process_alfrttm_file(
    input_file: Path,
    max_duration: float = 20.0,
    use_context_inference: bool = True
) -> tuple[Path, Path]:

    input_file = Path(input_file)

    with input_file.open(
        "r",
        encoding="utf-8-sig"
    ) as file:
        lines = file.readlines()

    segments = parse_segments(lines)

    segments = repair_segments(
        segments,
        max_duration=max_duration,
        use_context_inference=use_context_inference
    )

    corrected_file = input_file.with_name(
        f"corrected_{input_file.name}"
    )

    corrected_content = "\n".join(
        segment.corrected_line
        for segment in segments
    )

    # Retain a final newline.
    corrected_content += "\n"

    with corrected_file.open(
        "w",
        encoding="utf-8",
        newline="\n"
    ) as file:
        file.write(corrected_content)

    error_csv = write_error_report(
        input_file,
        segments
    )

    # Print the corrected ALFRTTM content.
    print("=" * 80)
    print(f"INPUT:     {input_file}")
    print(f"CORRECTED: {corrected_file}")
    print(f"REPORT:    {error_csv}")
    print("=" * 80)
    print(corrected_content)

    statuses = {}

    for segment in segments:
        statuses[segment.status] = (
            statuses.get(segment.status, 0) + 1
        )

    print("SUMMARY")
    for status, count in sorted(statuses.items()):
        print(f"{status}: {count}")

    return corrected_file, error_csv


def process_target(target: Path) -> None:
    """
    Process either one .alfrttm file or all .alfrttm files
    directly inside a folder.
    """
    target = Path(target)

    if target.is_file():
        # if target.suffix.lower() != ".alfrttm":
        #     raise ValueError(
        #         f"Not an .alfrttm file: {target}"
        #     )

        process_alfrttm_file(
            target,
            max_duration=MAX_DURATION,
            use_context_inference=USE_CONTEXT_INFERENCE
        )

    elif target.is_dir():
        files = sorted(target.glob("*.alfrttm"))

        # Do not reprocess files created by this script.
        files = [
            file for file in files
            if not file.name.startswith("corrected_")
        ]

        if not files:
            raise FileNotFoundError(
                f"No .alfrttm files found in {target}"
            )

        for file in files:
            process_alfrttm_file(
                file,
                max_duration=MAX_DURATION,
                use_context_inference=USE_CONTEXT_INFERENCE
            )

    else:
        raise FileNotFoundError(
            f"Input path does not exist: {target}"
        )


# ============================================================
# RUN
# ============================================================

process_target(INPUT_TARGET)